# GigaPath inference (prov-gigapath/prov-gigapath)
Quick notebook to run the gated GigaPath image feature extractor. You need Hugging Face access to the repo.

## 0. Install/upgrade deps
Uncomment if you need to install locally.

In [1]:
# !pip install -U 'torch>=2.1' timm huggingface-hub pillow openslide-python scikit-image tqdm


## 1. Auth and repo inspection
Set `HF_TOKEN` to a token that has access to `prov-gigapath/prov-gigapath` (accept the license first).

In [2]:
import os
from pathlib import Path
from huggingface_hub import login, HfApi

def _load_hf_token():
    # 우선 환경변수 확인
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if token:
        os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)
        return token
    # .env 후보 경로 탐색 (노트북 기준 ./, ../, ../../)
    candidates = [Path('.env'), Path('../.env'), Path('../../.env')]
    for path in candidates:
        if path.exists():
            for line in path.read_text().splitlines():
                if not line or line.strip().startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                if k.strip() in ('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
                    token = v.strip()
                    os.environ['HF_TOKEN'] = token
                    os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)
                    return token
    raise RuntimeError('HF_TOKEN을 환경변수로 설정하거나 .env에 HF_TOKEN=... 값을 추가하세요 (gated 모델 접근 필요).')

HF_TOKEN = _load_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi()
files = api.list_repo_files('prov-gigapath/prov-gigapath', repo_type='model', token=HF_TOKEN)
print('Repo files (first 30):')
for f in files[:30]:
    print(' -', f)
print(f'Total files: {len(files)}')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Repo files (first 30):
 - .gitattributes
 - README.md
 - config.json
 - pytorch_model.bin
 - sample_data/PROV-000-000001.ndpi
 - slide_encoder.pth
Total files: 6


## 2. Tile SVS slides from Data/2023
``.svs`` → PNG 타일로 변환 (256px, tissue mask 적용). 타일 경로가 `tile_paths`에 채워집니다.


In [3]:

import numpy as np
import openslide
from skimage import color, filters
from PIL import Image, ImageFile, PngImagePlugin
from tqdm import tqdm
from pathlib import Path

PngImagePlugin.MAX_TEXT_CHUNK = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

PATCH_SIZE = 256
MIN_TISSUE_RATIO = 0.6
LEVEL = 0
MAX_TILES = 500  # 테스트용으로 최대 500장만 추출

# 출력 루트는 현재 작업 디렉토리 기준 ./output_tiles
OUT_ROOT = Path.cwd() / 'output_tiles'

def tissue_mask(np_rgb: np.ndarray) -> np.ndarray:
    hsv = color.rgb2hsv(np_rgb)
    saturation = hsv[:, :, 1]
    thresh = filters.threshold_otsu(saturation)
    return saturation > thresh

def iter_tiles(slide: openslide.OpenSlide, stride: int = PATCH_SIZE):
    width, height = slide.level_dimensions[LEVEL]
    for y in range(0, height, stride):
        for x in range(0, width, stride):
            region = slide.read_region((x, y), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')
            arr = np.array(region)
            mask = tissue_mask(arr)
            if mask.mean() < MIN_TISSUE_RATIO:
                continue
            yield x, y, region

def export_tiles(slide_path: Path, out_dir: Path, max_tiles: int = MAX_TILES):
    out_dir.mkdir(parents=True, exist_ok=True)
    slide = openslide.OpenSlide(str(slide_path))
    meta = []
    width, height = slide.level_dimensions[LEVEL]
    est_tiles = ((height + PATCH_SIZE - 1) // PATCH_SIZE) * ((width + PATCH_SIZE - 1) // PATCH_SIZE)
    total_tiles = est_tiles
    last_pct = -1
    print(f'tile {slide_path.name}: start (est {total_tiles} tiles, cap {max_tiles})')
    print('  - 출력 예시: tile <파일명>: <진행%> (현재/총 타일 수)')
    for tile_idx, (x, y, tile_img) in enumerate(iter_tiles(slide)):
        pct = int((tile_idx + 1) * 100 / max(1, total_tiles))
        if pct != last_pct:
            print(f'tile {slide_path.name}: {pct}% ({tile_idx+1}/{total_tiles} tiles)')
            last_pct = pct
        tile_name = f"tile_{tile_idx:06d}_x{x}_y{y}.png"
        tile_path = out_dir / tile_name
        tile_img.save(tile_path, format='PNG')
        meta.append({'tile_path': str(tile_path), 'x': x, 'y': y, 'level': LEVEL})
        if len(meta) >= max_tiles:
            print(f'Max tiles reached: {max_tiles}; stopping early.')
            break
    slide.close()
    return meta

candidate_roots = [Path('Data/2023'), Path('../Data/2023'), Path('../../Data/2023')]
svs_list = []
for root in candidate_roots:
    if root.exists():
        svs_list.extend(sorted(root.glob('**/*.svs')))
if not svs_list:
    raise RuntimeError('Data/2023 경로(./, ../, ../../) 하위에서 .svs 파일을 찾지 못했습니다.')
print('Found SVS files:')
for i, p in enumerate(svs_list[:5]):
    print(f' [{i}] {p}')
slide_idx = 0  # 다른 슬라이드를 쓰려면 인덱스 변경
slide_path = svs_list[slide_idx]
tiles_dir = OUT_ROOT / slide_path.stem
meta = export_tiles(slide_path, tiles_dir, max_tiles=MAX_TILES)
tile_paths = [Path(m['tile_path']) for m in meta]
print(f'Total kept tiles: {len(tile_paths)} -> {tiles_dir}')


Found SVS files:
 [0] ../../Data/2023/S23-00049/S23-00049#1###4.svs
 [1] ../../Data/2023/S23-00161/S23-00161#1#A##7.svs
 [2] ../../Data/2023/S23-00188/S23-00188#1###9.svs
 [3] ../../Data/2023/S23-00233/S23-00233#1###9.svs
 [4] ../../Data/2023/S23-00284/S23-00284#1###0.svs
tile S23-00049#1###4.svs: start (est 11009 tiles, cap 500)
  - 출력 예시: tile <파일명>: <진행%> (현재/총 타일 수)
tile S23-00049#1###4.svs: 0% (1/11009 tiles)
tile S23-00049#1###4.svs: 1% (111/11009 tiles)
tile S23-00049#1###4.svs: 2% (221/11009 tiles)
tile S23-00049#1###4.svs: 3% (331/11009 tiles)
tile S23-00049#1###4.svs: 4% (441/11009 tiles)
Max tiles reached: 500; stopping early.
Total kept tiles: 500 -> /Users/curv/Repos/GC-Pathology/PoC/v1/output_tiles/S23-00049#1###4


## 3. Load model via timm hf_hub
timm can pull weights directly from the repo when you use the `hf_hub:` prefix.

In [4]:
import torch
import timm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'hf_hub:prov-gigapath/prov-gigapath'
model = timm.create_model(model_name, pretrained=True).to(device)
model.eval()
print('Loaded model on', device)

Loaded model on cpu


## 4. Run inference on tiles
타일 폴더(`tile_paths`)를 배치 단위로 돌며 특징을 추출합니다.


In [5]:

from pathlib import Path
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import torch

try:
    from torchvision.io import read_image
    from torchvision.transforms.functional import to_pil_image
    HAS_TORCHVISION = True
except ImportError:
    HAS_TORCHVISION = False
from PIL import Image, ImageFile, PngImagePlugin, UnidentifiedImageError

ImageFile.LOAD_TRUNCATED_IMAGES = True
PngImagePlugin.MAX_TEXT_CHUNK = None

if 'tile_paths' not in globals() or not tile_paths:
    raise RuntimeError('tile_paths가 비어 있습니다. 먼저 타일링 셀을 실행하세요.')

out_root = Path.cwd() / 'output_tiles'
_tile_paths = [Path(p) for p in tile_paths if Path(p).is_file()]
if not _tile_paths and out_root.exists():
    _tile_paths = sorted(out_root.glob('**/*.png'))
tile_paths = _tile_paths
print(f'Using {len(tile_paths)} tiles')

cfg = resolve_data_config({}, model=model)
transform = create_transform(**cfg)
batch_size = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'

features = []
skipped = 0

for i in range(0, len(tile_paths), batch_size):
    batch_paths = tile_paths[i:i+batch_size]
    imgs = []
    ok_paths = []
    for p in batch_paths:
        try:
            if HAS_TORCHVISION:
                # read_image -> CHW uint8 tensor; convert to PIL then timm transform
                t = read_image(str(p))
                img = to_pil_image(t)
            else:
                img = Image.open(p).convert('RGB')
            imgs.append(transform(img))
            ok_paths.append(p)
        except UnidentifiedImageError:
            skipped += 1
            print(f'Skipping unreadable image: {p}')
        except Exception as e:
            skipped += 1
            print(f'Skipping image due to error: {p} ({e})')
    if not imgs:
        continue
    x = torch.stack(imgs).to(device)
    with torch.no_grad():
        feats = model.forward_features(x)
        if isinstance(feats, (list, tuple)):
            feats = feats[-1]
        if feats.ndim == 4:
            pooled = feats.mean(dim=(2, 3))
        else:
            pooled = feats
    features.append(pooled.cpu())
    print(f'batch {i//batch_size + 1}: {len(ok_paths)} tiles -> {pooled.shape}')

feature_tensor = torch.cat(features) if features else torch.empty(0)
print('All features shape:', feature_tensor.shape)
print('Skipped images:', skipped)
print('First vector (5 dims):', feature_tensor[0, :5].numpy() if feature_tensor.numel() else 'n/a')


Using 500 tiles
batch 1: 16 tiles -> torch.Size([16, 197, 1536])
batch 2: 16 tiles -> torch.Size([16, 197, 1536])
batch 3: 16 tiles -> torch.Size([16, 197, 1536])
batch 4: 16 tiles -> torch.Size([16, 197, 1536])
batch 5: 16 tiles -> torch.Size([16, 197, 1536])
batch 6: 16 tiles -> torch.Size([16, 197, 1536])
batch 7: 16 tiles -> torch.Size([16, 197, 1536])
batch 8: 16 tiles -> torch.Size([16, 197, 1536])
batch 9: 16 tiles -> torch.Size([16, 197, 1536])
batch 10: 16 tiles -> torch.Size([16, 197, 1536])
batch 11: 16 tiles -> torch.Size([16, 197, 1536])
batch 12: 16 tiles -> torch.Size([16, 197, 1536])
batch 13: 16 tiles -> torch.Size([16, 197, 1536])
batch 14: 16 tiles -> torch.Size([16, 197, 1536])
batch 15: 16 tiles -> torch.Size([16, 197, 1536])
batch 16: 16 tiles -> torch.Size([16, 197, 1536])
batch 17: 16 tiles -> torch.Size([16, 197, 1536])
batch 18: 16 tiles -> torch.Size([16, 197, 1536])
batch 19: 16 tiles -> torch.Size([16, 197, 1536])
batch 20: 16 tiles -> torch.Size([16, 197, 

In [6]:
feature_tensor.shape

torch.Size([500, 197, 1536])

## 5. Optional: manual checkpoint download
Use this if `timm.create_model` cannot locate the right checkpoint; specify the filename after listing repo files.


In [ ]:
from huggingface_hub import hf_hub_download

# Example: ckpt_name = 'gigapath_base.pt'  # change to actual file name from the repo
ckpt_name = None
if ckpt_name:
    ckpt_path = hf_hub_download('prov-gigapath/prov-gigapath', filename=ckpt_name, repo_type='model', token=HF_TOKEN or None)
    print('Downloaded to', ckpt_path)
    # Load with torch.load or timm.load_checkpoint as appropriate
else:
    print('Set ckpt_name once you know the checkpoint filename from the repo.')